# Omnibus — dwell time as a demand proxy (Model B)

RVV gave us **no passenger counts**. But `dwell_s` (`ts_departure_actual_door − ts_arrival_actual_door`) is how long the doors were open — and door-open time is dominated by boardings/alightings. So dwell is a **proxy for demand** (PLAN.md → Model B). This notebook makes that proxy visible two ways:

1. **Where** demand concentrates — a map of mean dwell per stop.
2. **Crowded-slow vs traffic-slow** — dwell vs delay per stop. A stop that's slow *and* high-dwell is congested with people; slow *but* low-dwell loses its time moving between stops (traffic / signals). Different problems, different fixes — and you can't tell them apart from delay alone.

Caveats: Line-1 overlap dropped, `|delay_arr_s| < 7200`, door-opened only. **Crucially we strip schedule-holding, not just close-door events:** a bus's first/last stop is a terminus layover, and any dwell > 120 s is a timing-point hold where the driver waits to leave *on time* — neither is boarding demand. So: exclude each trip's first/last stop, cap dwell at 120 s, and use the **median** (robust to the few remaining holds). Without this the "demand map" just lights up the depots.

In [ ]:
import polars as pl
import folium
import branca.colormap as cmm
import matplotlib.pyplot as plt
import numpy as np

df = pl.read_parquet("../data/parquet/features.parquet")
clean = df.filter((pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")
                  & (pl.col("delay_arr_s").abs() < 7200))
base = clean.filter(pl.col("door_opened") & (pl.col("dwell_s") > 0)
                    & pl.col("stop_lat").is_not_null())
# strip terminus layovers (first/last stop of each trip — pre-computed `is_terminus`
# in assemble.py) and timing-point holds (>120s), leaving dwell that's plausibly
# boarding/alighting demand
dwell = base.filter(~pl.col("is_terminus") & (pl.col("dwell_s") <= 120))
print(f"boarding-dwell events with coords: {dwell.height:,}  "
      f"(dropped {base.height - dwell.height:,} terminus/hold events)")

## 1 — Demand map: mean dwell per stop

Bigger, redder = longer typical door-open time = more boarding activity. This is the closest thing we have to a ridership heatmap.

In [ ]:
per_stop = (dwell.group_by("stop_name").agg(
                pl.col("dwell_s").median().alias("dwell_med"),
                pl.col("dwell_s").mean().alias("dwell_mean"),
                pl.col("delay_arr_s").median().alias("delay_med"),
                pl.col("stop_lat").first().alias("lat"),
                pl.col("stop_lon").first().alias("lon"),
                pl.len().alias("n"))
            .filter(pl.col("n") > 500)
            .sort("dwell_med", descending=True))

dv = per_stop["dwell_med"].to_numpy()
vmin, vmax = float(np.percentile(dv, 5)), float(np.percentile(dv, 95))
cmap = cmm.LinearColormap(["#2c7fb8", "#fee08b", "#d73027"], vmin=vmin, vmax=vmax)
cmap.caption = "median dwell [s] — proxy for boarding demand (blue = quiet, red = busy)"

m = folium.Map(location=[49.0125, 12.0992], zoom_start=12, tiles="CartoDB positron")
for r in per_stop.iter_rows(named=True):
    folium.CircleMarker(
        [r["lat"], r["lon"]], radius=3 + (r["dwell_med"] - vmin) / (vmax - vmin) * 12,
        color="#333", weight=0.4, fill=True, fill_color=cmap(r["dwell_med"]),
        fill_opacity=0.82,
        tooltip=f"{r['stop_name']} — dwell {r['dwell_med']:.0f}s, n={r['n']:,}",
        popup=folium.Popup(f"<b>{r['stop_name']}</b><br>median dwell {r['dwell_med']:.0f}s"
                           f"<br>median delay {r['delay_med']:.0f}s<br>samples {r['n']:,}", max_width=260),
    ).add_to(m)
cmap.add_to(m)
print("top demand stops:")
print(per_stop.head(8).select("stop_name", "dwell_med", "delay_med", "n"))
m

**How to read it.** The red cluster lands on the obvious generators — **Hauptbahnhof** (median 40 s, far ahead of everything), then **Universität**, **Arnulfsplatz**, **HBF Süd/Arcaden**, **Bismarckplatz**, **TechCampus/OTH**. That's the sanity check that dwell really is tracking ridership, not schedule artefacts. Quiet blue stops on the outer ring are low-boarding request stops. For the pitch this *is* the ridership map RVV said we couldn't have — reconstructed from door timings. (Note Klinikum is *not* a dwell hotspot — its delay problem is congestion getting *to* it, not boarding volume; the next chart separates exactly that.)

## 2 — Crowded-slow vs traffic-slow

Plot every stop as (mean dwell, median delay). The two axes decompose "slow":
- **top-right** = busy *and* late → demand-driven congestion (more service / dwell-time signal priority).
- **top-left** = late but *not* busy → time lost between stops → traffic / signals (corridor fix).
- **bottom** = on time → leave alone.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
x = per_stop["dwell_med"].to_numpy(); y = per_stop["delay_med"].to_numpy()
sizes = per_stop["n"].to_numpy() / 400
sc = ax.scatter(x, y, s=sizes, c=y, cmap="RdYlGn_r", alpha=0.75,
                edgecolor="black", linewidth=0.3)
xm, ym = np.median(x), np.median(y)
ax.axvline(xm, color="#888", ls="--", lw=1); ax.axhline(ym, color="#888", ls="--", lw=1)
ax.text(0.98, 0.97, "traffic-slow\n(late, not busy)", transform=ax.transAxes,
        ha="right", va="top", fontsize=9, color="#444")
ax.text(0.98, 0.03, "demand-driven\n(busy & late)", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9, color="#444")

# label the extreme stops in each quadrant
busy_late = per_stop.filter((pl.col("dwell_med") > xm) & (pl.col("delay_med") > ym)).sort("delay_med", descending=True).head(6)
late_quiet = per_stop.filter((pl.col("dwell_med") < xm) & (pl.col("delay_med") > ym)).sort("delay_med", descending=True).head(6)
for r in pl.concat([busy_late, late_quiet]).iter_rows(named=True):
    ax.annotate(r["stop_name"], (r["dwell_med"], r["delay_med"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("median dwell [s]  →  more boarding demand")
ax.set_ylabel("median arrival delay [s]  →  later")
ax.set_title("Crowded-slow vs traffic-slow — quadrants point to different interventions")
plt.colorbar(sc, label="median delay [s]"); ax.grid(True, alpha=0.25)
fig.tight_layout(); plt.show()

**How to read it.** Stops in the **top-left** (late but low dwell) are where buses bleed time *driving* — these are the candidates for transit-signal priority or a bus lane, and they map straight onto the σ-hotspot corridor from notebook 04. Stops in the **top-right** (late and high dwell) need capacity, not asphalt — more frequent service or all-door boarding. The decomposition is only possible because we have door-level dwell; it's the analytical core of Model B and a clean "we found *why*, not just *where*" beat for the pitch.

## 3 — When demand moves: the city breathing

The maps above are time-averaged — they show *where* demand sits, not *when*. The flex-routing pitch lives on the time axis: demand isn't static, it **migrates** across the day. `demand_surface.parquet` (built by `pipeline/build_demand.py`, same boarding-dwell cleaning as above — terminus stops dropped, dwell capped at 120 s, baseline = the two normal October full-network windows) aggregates dwell to a `(stop × daytype × hour)` grid, normalised to a *per-typical-day* figure so the 2-week sample size cancels out.

First view: the whole city's pulse by hour, split by daytype.

In [ ]:
surf = pl.read_parquet("../data/parquet/demand_surface.parquet")
print(f"demand surface: {surf.height:,} cells, {surf['stop_code'].n_unique()} stops, "
      f"daytypes {surf['daytype'].unique().to_list()}")

# city-wide boarding-dwell per typical day, by hour and daytype
city = (surf.group_by("daytype", "hour")
            .agg(pl.col("dwell_per_day_s").sum().alias("total"))
            .sort("hour"))

fig, ax = plt.subplots(figsize=(11, 5))
for dt, colour in [("weekday", "#d73027"), ("sat", "#fc8d59"), ("sun", "#4575b4")]:
    s = city.filter(pl.col("daytype") == dt).sort("hour")
    ax.plot(s["hour"], s["total"], marker="o", ms=4, label=dt, color=colour)
ax.set_xlabel("hour of day"); ax.set_ylabel("city-wide boarding-dwell per typical day [s]")
ax.set_title("Regensburg's demand pulse — when the city moves")
ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## 4 — Which stops own which hours

The line above is the *aggregate* pulse. This heatmap breaks it apart per stop. Each row is a stop, each column an hour; colour is that stop's boarding-dwell **normalised to its own daily peak** — so we read *timing*, not magnitude (Hbf would otherwise drown everyone). Banding by row reveals the migration the flex-routing pitch exploits: commuter stops fire at 07–08, the campuses (Universität, TechCampus/OTH) hold a midday plateau, residential stops light up on the evening return. Where two stops peak in the *same* off-peak hour but the timetable serves them thinly → that's a flex-route candidate.

In [ ]:
# stop x hour heatmap (weekday), row-normalized so we see *when* each stop peaks
wk = surf.filter(pl.col("daytype") == "weekday")
top = (wk.group_by("stop_name").agg(pl.col("dwell_per_day_s").sum().alias("tot"))
         .sort("tot", descending=True).head(25))
top_names = top["stop_name"].to_list()

cell = {(r["stop_name"], r["hour"]): r["dwell_per_day_s"] for r in wk.iter_rows(named=True)}
M = np.array([[cell.get((name, h), 0.0) for h in range(24)] for name in top_names])
peak = M.max(axis=1, keepdims=True)
rown = M / np.where(peak == 0, 1, peak)  # share of each stop's own daily peak

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(rown, aspect="auto", cmap="magma")
ax.set_xticks(range(24)); ax.set_xticklabels(range(24))
ax.set_yticks(range(len(top_names))); ax.set_yticklabels(top_names, fontsize=8)
ax.set_xlabel("hour of day")
ax.set_title("When each stop peaks — row-normalized boarding-dwell (weekday, top 25 stops)")
fig.colorbar(im, ax=ax, label="share of stop's own daily peak")
plt.tight_layout(); plt.show()

## 5 — The city breathing, on the map

The two views above split space and time; this fuses them. An animated heatmap over Regensburg, one frame per weekday hour (05:00 → 23:00). Hit play (or scrub the slider) and watch demand **move**: morning bloom on the commuter corridors and the Hbf, midday weight shifting onto the campuses and the Altstadt, an evening surge on the return stops.

Intensity is **log-scaled** against a single global maximum (`use_local_extrema=False`). Hbf is so far ahead of every other stop that a linear scale leaves it the only visible thing on the map — log compresses that range so the mid-tier stops actually render, while Hbf stays the brightest and quiet hours stay genuinely faint (the breathing survives). This is the in-notebook proof of the demo's centerpiece; the frontend map (layer 4) is the production version of exactly this.

In [ ]:
from folium.plugins import HeatMapWithTime

wk = surf.filter(pl.col("daytype") == "weekday")
hours = list(range(5, 24))  # skip the dead 00-04 window

# log1p scaling: Hbf dwarfs everything on a linear scale, so every other stop
# falls below the colour floor and vanishes. log compresses the range -> small
# stops lift into the visible gradient while Hbf stays brightest. Global denom
# (log of the busiest cell) keeps hours comparable: quiet hours stay fainter.
denom = np.log1p(wk["dwell_per_day_s"].max())

frames = []
for h in hours:
    f = wk.filter(pl.col("hour") == h)
    frames.append([[r["stop_lat"], r["stop_lon"], float(np.log1p(r["dwell_per_day_s"]) / denom)]
                   for r in f.iter_rows(named=True)])

m2 = folium.Map(location=[49.0125, 12.0992], zoom_start=12, tiles="CartoDB positron")
HeatMapWithTime(
    frames, index=[f"{h:02d}:00" for h in hours],
    radius=16, blur=0.7, max_opacity=0.9, min_opacity=0.35, use_local_extrema=False,
    # gradient floor near 0 so even low-weight stops get a colour, not transparency
    gradient={0.05: "#2c7fb8", 0.35: "#41b6c4", 0.6: "#fee08b", 0.8: "#fc8d59", 1.0: "#d73027"},
    auto_play=True, display_index=True,
).add_to(m2)
print("drag the time slider (or hit play) — weekday demand 05:00 -> 23:00 (log-scaled)")
m2